<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/dynatree-cre/notebooks/m7_5_adapter_colab_seed2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M7.5 Step 4 — Learned Contrastive Query Adapter (`DT-C0.6a/b` & `DT-C4.6`, Colab GPU, Seed 2021)
- **Part A (~3–4 phút cho cả 5 lượt):**
  - `DT-C0.6a` (`t1_frozen`, zero pseudo-replay, zero drift) trên FewRel.
  - `DT-C0.6b` (`continual_anchor_preserved`, `drift_weight=5.0` vs control `drift_weight=0.0`) trên FewRel & TACRED — đo trực tiếp `old_ret`, `new_ret` và `representation_drift_l2`.
- **Part B:** `DT-C4.6` (Tree + Continual Adapter + Candidate Reranking `--rerank-sweep`) trên FewRel Std, Exp B (`semantic_homogeneous`), Exp C (`cross_task_dispersed`) & TACRED.
- **Part C:** Tổng hợp bảng & tải về `m7_5_adapter_colab_results.zip`.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Chon GPU runtime (T4/L4/A100)!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
BRANCH = 'feat/dynatree-cre'
REPO_DIR = '/content/research-code-lab'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scipy scikit-learn tabulate pytest
!python scripts/setup_protocol_v1_data.py

gdrive_runs = Path('/content/drive/MyDrive/TAPTA_DynaTree_CRE/runs/dynatree_cre')
gdrive_runs.mkdir(parents=True, exist_ok=True)
local_runs = Path('/content/research-code-lab/runs/dynatree_cre')
local_runs.parent.mkdir(parents=True, exist_ok=True)
if local_runs.is_symlink():
    local_runs.unlink()
elif local_runs.exists():
    shutil.copytree(local_runs, gdrive_runs, dirs_exist_ok=True)
    shutil.rmtree(local_runs)
local_runs.symlink_to(gdrive_runs)
!pytest tests/dynatree_cre -q
print('Setup & Drive backup ready ->', gdrive_runs)

In [ ]:
# Part A — Standalone Adapter Retrieval Controls (FewRel + TACRED)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c0_6a_t1_adapter_fewrel_5shot_seed2021.yaml \
    --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c0_6b_continual_adapter_fewrel_5shot_seed2021.yaml \
    --device cuda --allow-hub-download

# Control ablation: Continual Adapter WITHOUT drift loss (drift_weight=0.0)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c0_6b_continual_adapter_fewrel_5shot_seed2021.yaml \
    --query-adapter-drift-weight 0.0 \
    --output-dir runs/dynatree_cre/m7_5_dt_c0_6b_nodrift/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download

# TACRED DT-C0.6a & DT-C0.6b controls
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c4_6_tree_adapter_rerank_tacred_5shot_seed2021.yaml \
    --routing-strategy none --inference-mode soft_mix --epochs-override 0 \
    --query-adapter-mode t1_frozen \
    --output-dir runs/dynatree_cre/m7_5_dt_c0_6a_t1_adapter/tacred_5shot/seed_2021 \
    --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c4_6_tree_adapter_rerank_tacred_5shot_seed2021.yaml \
    --routing-strategy none --inference-mode soft_mix --epochs-override 0 \
    --output-dir runs/dynatree_cre/m7_5_dt_c0_6b_continual_adapter/tacred_5shot/seed_2021 \
    --device cuda --allow-hub-download

In [ ]:
# Part B — DT-C4.6 (Tree + Continual Adapter + Candidate Reranking) tren FewRel Std, Exp B, Exp C & TACRED
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c4_6_tree_adapter_rerank_fewrel_5shot_seed2021.yaml \
    --rerank-sweep --device cuda --allow-hub-download

for dist, tag in [('semantic_homogeneous', 'expB'), ('cross_task_dispersed', 'expC')]:
    !python experiments/run_dynatree_cre.py \
        --config configs/dynatree_cre/m7_5_dt_c4_6_tree_adapter_rerank_fewrel_5shot_seed2021.yaml \
        --task-distribution {dist} \
        --output-dir runs/dynatree_cre/m7_5_dt_c4_6_{tag}/fewrel_5shot/seed_2021 \
        --rerank-sweep --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c4_6_tree_adapter_rerank_tacred_5shot_seed2021.yaml \
    --rerank-sweep --device cuda --allow-hub-download

In [ ]:
# Part C — Summary Table & Export m7_5_adapter_colab_results.zip
import json
import os
import shutil
from pathlib import Path
from tabulate import tabulate
from google.colab import files

runs_root = Path('/content/research-code-lab/runs/dynatree_cre')
bundle = Path('/content/m7_5_adapter_bundle')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True)

rows = []
for m_path in sorted(runs_root.glob('m7_5_dt_c*6*/**/metrics.json')):
    run_dir = m_path.parent
    rel = run_dir.relative_to(runs_root)
    shutil.copytree(run_dir, bundle / rel, ignore=shutil.ignore_patterns('*.pt'), dirs_exist_ok=True)
    d = json.loads(m_path.read_text())
    res = json.loads((run_dir / 'resource_metrics.json').read_text())
    ovn = d['stages'][-1]['eval_stats'].get('retrieval_old_vs_new', {})
    sw_p = run_dir / 'rerank_sweep.json'
    best_s = 'N/A'
    if sw_p.exists():
        sw = json.loads(sw_p.read_text())
        bc = max(sw['cells'], key=lambda c: c['final_ncm_whole_accuracy'])
        best_s = f"{bc['final_ncm_whole_accuracy']*100:.2f}% (K={bc['k']}, {bc['fusion_mode']})"
    rows.append([
        rel.parts[0],
        d.get('dataset'),
        d.get('task_distribution'),
        f"{d.get('final_ncm_whole_accuracy', 0)*100:.2f}%",
        f"{d.get('ncm_summary', {}).get('average_forgetting', 0)*100:.2f}%",
        f"{(ovn.get('retrieval_old_top1_acc') or 0)*100:.2f}%",
        f"{(ovn.get('retrieval_new_top1_acc') or 0)*100:.2f}%",
        f"{ovn.get('representation_drift_l2', 0):.4f}",
        best_s,
        f"{res['memory_bytes']['total_persistent_state_bytes']/(1024**2):.2f} MB",
    ])

headers = ['Run', 'Dataset', 'Dist', 'Main NCM', 'NCM F', 'OldRet@1', 'NewRet@1', 'DriftL2', 'Best Sweep', 'Mem']
print(tabulate(rows, headers=headers, tablefmt='grid'))
shutil.make_archive('/content/m7_5_adapter_colab_results', 'zip', bundle)
print('Bundle size KB:', os.path.getsize('/content/m7_5_adapter_colab_results.zip') / 1024)
files.download('/content/m7_5_adapter_colab_results.zip')